# AmbiFix — Back-Translation روی مک M3 Pro

این نوت‌بوک فقط داده‌های **Train** را Back-Translate می‌کند و `parent_text` و `source_id` را نگه می‌دارد. Test در بسته نیست؛ Validation فقط برای کنترل هم‌پوشانی استفاده می‌شود. همهٔ برچسب‌های مصنوعی هنوز **تأییدنشده** هستند.

Corrected local working copy. The immutable supplied notebook remains in `source_package/`. Use the experiment virtual environment and recorded lock files. Actual execution evidence is in `../reports/`; this notebook itself remains unexecuted.

## 1 — نصب وابستگی‌ها

ترجیحاً با محیط مجازی پروژه اجرا کن. دانلود مدل‌های ترجمه به اینترنت نیاز دارد. اگر PyTorch قبلاً نصب است، نیازی به نصب مجدد نیست. پس از نصب اولیه اگر خواست، کرنل را Restart کن.

In [ ]:
# Optional only, inside the isolated experiment kernel:
%pip install -r ../config/requirements-lock.txt

## 2 — تشخیص MPS و بررسی داده‌ها

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
from pathlib import Path
assert Path("scripts/backtranslate_with_lineage.py").exists(), "Jupyter را از پوشهٔ اصلی ZIP باز کن"
import torch
print("PyTorch:", torch.__version__)
print("Apple Metal GPU (MPS):", torch.backends.mps.is_available())
print("If False, script falls back to CPU.")
from scripts.backtranslate_with_lineage import main
main(["--dry-run"])

## 3 — اجرای ترجمه

پیش‌فرض ۹۰۰ نمونهٔ اصلی+مصنوعی برای هر کلاس ابهام است و کلاس Clean افزایش پیدا نمی‌کند. کد قابلیت Resume دارد؛ با اجرای دوبارهٔ همین سلول، کار از ژورنال ذخیره‌شده ادامه پیدا می‌کند. برای حافظهٔ کمتر `batch-size=1` انتخاب کن. برای اجبار CPU، مقدار `auto` را به `cpu` تغییر بده.

In [ ]:
main([
    "--target-per-class", "900",
    "--pivots", "fr", "de", "es",
    "--variants-per-pivot", "1",
    "--batch-size", "2",
    "--max-input-tokens", "192", "--max-output-tokens", "192",
    "--seed", "42", "--device", "auto",
    "--model-lock", "../config/model_lock.json",
    "--output", "../runs/full",
])

## 4 — بررسی و بسته‌بندی خروجی

خروجی شامل `AUDIT.json`، داده‌های مصنوعی، جملهٔ مادر و ژورنال Resume است. نمونه‌های تولیدی هنوز تأیید مستقل برچسب ندارند.

In [ ]:
import json
from pathlib import Path
out = Path("../runs/full")
s = json.loads((out/"AUDIT.json").read_text(encoding="utf-8"))
for k in ["synthetic_accepted_count", "provisional_train_count", "per_class_train_provisional", "target_unmet", "verified_ambiguity_labels"]:
    print(k, ":", s[k])
print("Use the independently audited delivery ZIP in ../deliverables/; do not zip protected source directories.")